# 최근 K개 turn만 모델에 전달하기

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

과거의 `ConversationBufferWindowMemory(k=...)` 대신, 현재는 다음 두 책임을 분리하는
편이 안전합니다.

1. checkpointer에는 전체 스레드 상태를 보존합니다.
2. 모델을 호출하기 직전에 필요한 최근 turn만 선택합니다.

이렇게 하면 감사·디버깅에 필요한 원본 상태는 유지하면서 모델 비용과 산만함을
줄일 수 있습니다. 여기서는 **완료된 최근 2개 turn + 현재 질문**만 모델에 전달합니다.


In [ ]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" python-dotenv


In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 다른 공급자를 쓸 때는 예: anthropic:claude-... 처럼 지정할 수 있습니다.
MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.4-mini")
model = init_chat_model(MODEL_ID)


In [ ]:
from langchain.messages import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

K = 2
SYSTEM_PROMPT = "당신은 친절한 은행 상담 챗봇입니다."


def context_for_model(messages):
    """완료된 최근 K개 turn과 현재 사용자 메시지를 선택합니다."""
    if not messages or messages[-1].type != "human":
        raise ValueError("마지막 메시지는 현재 사용자의 human 메시지여야 합니다.")

    history, current_user_message = messages[:-1], messages[-1]
    recent_history = history[-2 * K :]  # human/ai 한 쌍이 1 turn
    return [
        SystemMessage(content=SYSTEM_PROMPT),
        *recent_history,
        current_user_message,
    ]


def call_model(state: MessagesState):
    response = model.invoke(context_for_model(state["messages"]))
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_edge(START, "model")
builder.add_edge("model", END)
window_chat = builder.compile(checkpointer=InMemorySaver())


## 여러 turn 실행하기

매 호출에는 새 메시지만 넘기지만, checkpointer가 같은 `thread_id`의 전체 메시지를
노드에 복원합니다.


In [ ]:
config = {"configurable": {"thread_id": "window-demo"}}

questions = [
    "비대면 계좌 개설을 시작하려면 무엇이 필요한가요?",
    "신분증을 준비했습니다. 다음 단계는 무엇인가요?",
    "본인 인증까지 마쳤습니다. 이제 무엇을 하나요?",
    "이용 약관에도 동의했습니다. 마지막 단계는 무엇인가요?",
    "가장 최근 안내만 짧게 다시 말해 주세요.",
]

for question in questions:
    result = window_chat.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    print(f"Q: {question}\nA: {result['messages'][-1].content}\n")


## 저장된 전체 기록과 실제 모델 입력 비교

아래에서 checkpoint에는 모든 turn이 남아 있지만, 마지막 호출에 전달된 컨텍스트는
제한되어 있음을 확인할 수 있습니다.


In [ ]:
all_messages = window_chat.get_state(config).values["messages"]
last_call_context = context_for_model(all_messages[:-1])  # 마지막 AI 응답 직전 상태

print("checkpoint에 저장된 메시지 수:", len(all_messages))
print("마지막 모델 호출에 전달된 메시지 수:", len(last_call_context))
print()

for message in last_call_context:
    print(f"[{message.type}] {message.content}")


상태에서도 오래된 메시지를 영구 삭제해야 한다면 `RemoveMessage`를 반환하는 별도
노드를 둘 수 있습니다. 다만 단순한 컨텍스트 절약 목적이라면 위처럼 **저장과 모델 입력을
분리**하는 방식이 보통 더 유연합니다.
